# Classification Lithologique Automatisée avec TerraTorch et Prithvi EO

## Introduction
Ce notebook présente une méthodologie avancée pour la classification lithologique à l'aide du modèle de fondation **Prithvi EO v2** de la NASA et d'IBM. Contrairement aux approches classiques qui se limitent aux pixels, nous exploitons ici l'intelligence contextuelle de l'IA pour identifier les signatures rocheuses complexes.

## Objectifs
*   **Extraction sémantique** : Utiliser le transformeur Prithvi pour extraire des vecteurs de caractéristiques (embeddings) profonds.
*   **Classification par densité** : Appliquer l'algorithme **HDBSCAN** pour isoler les unités lithologiques cohérentes tout en excluant le bruit (végétation dense, nuages résiduels).
*   **Cartographie de précision** : Générer une carte des affleurements rocheux identifiés sur la zone d'étude au Congo.

## Méthodologie
1.  **Setup** : Installation de TerraTorch et initialisation de Google Earth Engine.
2.  **Acquisition** : Téléchargement d'images Sentinel-2 corrigées (SR) via GEE.
3.  **Inférence** : Passage des 6 bandes spectrales dans l'encodeur Prithvi pour obtenir des descripteurs de texture et de minéralogie.
4.  **Segmentation** : Clustering HDBSCAN pour regrouper les signatures spectrales similaires en classes lithologiques.

In [ ]:
# ====================================================
# ÉTAPE 1 : Installation et Initialisation
# ====================================================
!pip install geemap earthengine-api scikit-learn rasterio geopandas shapely folium matplotlib seaborn terratorch torch -q

import ee, geemap, torch, rasterio, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from terratorch import BACKBONE_REGISTRY

try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

shared_tab20_colors = [plt.cm.tab20(i) for i in range(20)]
print('✅ Environnement prêt et couleurs initialisées')

## Définition de la Zone d'Étude (ROI)
Nous utilisons une zone d'intérêt au Congo, caractérisée par une diversité géologique intéressante, pour valider notre classification.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI (Congo)
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données Satellite
Nous récupérons une mosaïque Sentinel-2 médiane pour minimiser l'impact des nuages et obtenir une signature spectrale stable pour la lithologie.

In [ ]:
# ====================================================
# ÉTAPE 3 : Acquisition des données Sentinel-2
# ====================================================
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate('2023-01-01', '2023-12-31')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

image = collection.median().clip(roi)
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12'] # Bleu, Vert, Rouge, NIR, SWIR1, SWIR2
image_export = image.select(bands)

geemap.ee_export_image(image_export, 'input.tif', scale=30, region=roi)
print('✅ Données satellite exportées')

## Inférence avec Prithvi EO v2
Le modèle Prithvi convertit l'image brute en un espace de caractéristiques de haute dimension où les propriétés géologiques sont amplifiées.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence IA (TerraTorch)
# ====================================================
model = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', num_frames=1, in_chans=6, pretrained=True)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

with rasterio.open('input.tif') as src:
    img = src.read().astype(np.float32) / 10000.0
    profile = src.profile

input_tensor = torch.from_numpy(img).unsqueeze(0).to(device)
with torch.no_grad():
    output = model(input_tensor)
    features = output[0] if isinstance(output, list) else output
    
if len(features.shape) == 3:
    N = features.shape[1]
    side = int(np.sqrt(N-1))
    feature_grid = features[0, 1:].cpu().numpy().reshape(side, side, -1)
else:
    feature_grid = features[0].cpu().numpy().transpose(1, 2, 0)
print(f'✅ Caractéristiques extraites : {feature_grid.shape}')

## Analyse et Segmentation Lithologique
Nous appliquons HDBSCAN sur les descripteurs IA. L'avantage d'HDBSCAN est qu'il détecte automatiquement le nombre de classes de roches et ignore les pixels qui ne correspondent pas à une unité stable (marqués en noir/gris comme bruit).

In [ ]:
# ====================================================
# ÉTAPE 5 : Clustering et Visualisation
# ====================================================
h, w, d = feature_grid.shape
clusterer = HDBSCAN(min_cluster_size=15)
labels = clusterer.fit_predict(feature_grid.reshape(-1, d))
litho_map = labels.reshape(h, w)

plt.figure(figsize=(12, 10))
plt.imshow(litho_map, cmap='nipy_spectral')
plt.colorbar(label='Classes Lithologiques (-1 = Bruit/Végétation)')
plt.title('Carte Lithologique Automatisée (Prithvi + HDBSCAN)')
plt.axis('off')
plt.show()